# Step 9 — EgoVLP Training (MLP & Transformer)

Trains MLP and Transformer classifiers on the 256-dim EgoVLP features extracted in Step 6.

**Prerequisites:** EgoVLP features must be extracted in `FEATURES_DIR_EGOVLP`  
**Output:** checkpoints in `MY_MODELS_DIR/error_recognition/{MLP,Transformer}/egovlp/`

In [13]:
# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [14]:
# ── 2. Path constants (FIXED — do not modify) ─────────────────────────────────
DRIVE_ROOT          = '/content/drive/MyDrive/AML_Project'
FEATURES_DIR_EGOVLP = f'{DRIVE_ROOT}/CaptainCook4D/features_egovlp_num_frames_16'
MY_MODELS_DIR       = f'{DRIVE_ROOT}/models'
REPO_DIR            = '/content/code'
print('Paths defined.')

Paths defined.


In [15]:
# ── 3. Clone  ────────────────────────────────────────────────────
!git clone --recursive https://github.com/Laio95/aml-2025-mistake-detection.git {REPO_DIR}

fatal: destination path '/content/code' already exists and is not an empty directory.


In [16]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
!pip install -r {REPO_DIR}/requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118


In [17]:
# ── 5. Copying  ───────────────────────────────────────────────────
!mkdir code/data
!mkdir code/data/video
!mkdir code/data/video/egovlp

!cp -r {FEATURES_DIR_EGOVLP}/* code/data/video/egovlp
print("Copy complete.")

mkdir: cannot create directory ‘code/data’: File exists
mkdir: cannot create directory ‘code/data/video’: File exists
mkdir: cannot create directory ‘code/data/video/egovlp’: File exists
Copy complete.


In [18]:
# ── 6. Login wandb ───────────────────────────────────────────────────
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: albertogiunti2001 (albertogiunti2001-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Training — MLP

Hyperparameters (paper Appendix B.3, non-sequential models):
- `lr=1e-3`, `weight_decay=1e-3`, `num_epochs=50`, `pos_weight=1.5` (fixed in `base.py`)

In [ ]:
# ── 7. Train MLP — step split ─────────────────────────────────────────────────
%%bash

cd code

python train_er.py \
    --variant   MLP \
    --backbone  egovlp \
    --split     step \
    --num_epochs 50 \
    --lr        1e-3 \
    --batch_size 512 \
    --ckpt_directory {MY_MODELS_DIR}

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ── 8. Train MLP — recordings split ──────────────────────────────────────────
%%bash

cd code


python train_er.py \
    --variant   MLP \
    --backbone  egovlp \
    --split     recordings \
    --num_epochs 50 \
    --lr        1e-3 \
    --batch_size 512 \
    --ckpt_directory {MY_MODELS_DIR}

Output hidden; open in https://colab.research.google.com to view.

## Training — Transformer (ErFormer)

Same hyperparameters as MLP.  
Note: with 256-dim EgoVLP input, the modality-split branches in `ErFormer.forward()`
(`if dim // 1024 == 1`) are never triggered — the full 256-dim encoding goes
straight to the MLP decoder. This is correct behaviour for a single-modality backbone.

In [ ]:
# ── 9. Train Transformer — step split ────────────────────────────────────────
%%bash

cd code

python train_er.py \
    --variant   Transformer \
    --backbone  egovlp \
    --split     step \
    --num_epochs 50 \
    --lr        1e-5 \
    --batch_size 1 \
    --ckpt_directory {MY_MODELS_DIR}

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ── 10. Train Transformer — recordings split ───────────────────────────────────
%%bash

cd code

python train_er.py \
    --variant   Transformer \
    --backbone  egovlp \
    --split     recordings \
    --num_epochs 50 \
    --lr        1e-5 \
    --batch_size 1 \
    --ckpt_directory {MY_MODELS_DIR}

## Quick eval — verify checkpoints

Run after each training cell to check the best checkpoint was saved.
Full comparative analysis (vs Omnivore) is in Step 10.

In [ ]:
# ── 11. List saved checkpoints ────────────────────────────────────────────────
import pathlib
ckpt_root = pathlib.Path(MY_MODELS_DIR) / 'error_recognition'
for p in sorted(ckpt_root.rglob('*egovlp*_best.pt')):
    print(p)

In [ ]:
# ── 12. Quick global eval — MLP step split ────────────────────────────────────
MLP_STEP_CKPT = f'{MY_MODELS_DIR}/error_recognition/MLP/egovlp/error_recognition_step_egovlp_MLP_video_best.pt'

!python -m core.evaluate \
    --variant   MLP \
    --backbone  egovlp \
    --split     step \
    --threshold 0.6 \
    --segment_features_directory {FEATURES_DIR_EGOVLP} \
    --ckpt      {MLP_STEP_CKPT}

In [ ]:
# ── 13. Quick global eval — Transformer step split ───────────────────────────
TR_STEP_CKPT = f'{MY_MODELS_DIR}/error_recognition/Transformer/egovlp/error_recognition_step_egovlp_Transformer_video_best.pt'

!python -m core.evaluate \
    --variant   Transformer \
    --backbone  egovlp \
    --split     step \
    --threshold 0.6 \
    --segment_features_directory {FEATURES_DIR_EGOVLP} \
    --ckpt      {TR_STEP_CKPT}